---
title: 3.8.c Riesgo — Sismo y continuidad de infraestructura
short_title: c. Riesgo
---

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mromerot/sincelejo/blob/main/book/03_analisis/38_sismo/c_riesgo.ipynb)

**Reproducible sin red en el checkout.** La fuente preservada y sus limitaciones se describen en el capítulo 2.3.


In [1]:
from pathlib import Path
import sys, subprocess, json
candidatos = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidatos if (p/'src/sincelejo/analisis.py').exists()), None)
if ROOT is None:
    ROOT = Path('/content/sincelejo')
    if not ROOT.exists():
        subprocess.run(['git','clone','https://github.com/mromerot/sincelejo.git',str(ROOT)],check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(ROOT)+'[libro]'],check=True)
sys.path.insert(0,str(ROOT/'src'))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sincelejo.analisis import cargar, indicadores, deficit_trimestral, sensibilidad_calor
clima = cargar(ROOT/'data/raw/power_1991_2025.json')
anual = indicadores(clima)
evidencia = pd.read_json(ROOT/'data/evidencia.json')
plt.rcParams.update({'figure.figsize':(9,4), 'axes.spines.top':False, 'axes.spines.right':False})


## Resultado de riesgo que puede sostenerse

Se establece el protocolo y el inventario requerido. No se publica PAE sísmica ni pérdidas de un escenario ficticio. La falta de registros incorporados no demuestra amenaza nula.

In [2]:
local = evidencia.loc[evidencia.amenaza.eq('sismo'),['id','fecha','lugar','hallazgo','limite']]
if local.empty:
    print('No hay un hecho local incorporado en el registro para esta amenaza. No equivale a ausencia de riesgo.')
else:
    display(local)
censo=json.loads((ROOT/'data/censo2018.json').read_text())
print('Contexto demográfico: ',censo['total'],'personas censadas en 2018; no son personas expuestas a esta amenaza.')

No hay un hecho local incorporado en el registro para esta amenaza. No equivale a ausencia de riesgo.
Contexto demográfico:  274622 personas censadas en 2018; no son personas expuestas a esta amenaza.


## Exposición

Edificios residenciales, centros de salud, colegios, tanques, estaciones de bombeo y puentes. Para continuidad importa la dependencia entre componentes, además del costo de cada uno.

## Vulnerabilidad y capacidades

Sistema estructural, irregularidades, mantenimiento y elementos no estructurales. Una instalación que conserva su estructura puede quedar fuera de servicio por equipos, tuberías o energía.

In [3]:
display(pd.DataFrame([{'métrica_objetivo':'PGA/SA y fragilidad por tipología + daño, funcionalidad y PAE','datos_faltantes':'Curvas de amenaza, suelo, inventario estructural, costos y fragilidades compatibles','estado':'No estimada en esta edición'}]))

,métrica_objetivo,datos_faltantes,estado
0,"PGA/SA y fragilidad por tipología + daño, func...","Curvas de amenaza, suelo, inventario estructur...",No estimada en esta edición


## Ruta para cuantificar

Definir unidad y periodo; obtener la intensidad en cada ubicación; contrastar el inventario; validar la relación intensidad–impacto; y propagar incertidumbre. La cifra demográfica agregada no sustituye la intersección espacial. Las pérdidas directas y los efectos por interrupción se reportarán por separado para evitar doble conteo.

**Decisión actual:** Evaluar vulnerabilidad de instalaciones esenciales.